# Tool 3bis: Automatic sleep scoring (GSSC)

Scores every 30-s epoch of a night automatically with **GSSC** (Greifswald Sleep Stage Classifier, a
neural network trained on several thousand clinical PSG nights), for two uses:

- recordings **without an expert hypnogram**: the automatic hypnogram can stand in for it,
- recordings **with an expert hypnogram**: the two are compared (confusion matrix, Cohen's kappa,
  agreement per stage).

For each epoch GSSC also gives the **probability of each stage** (W, N1, N2, N3, R): the
**hypnodensity**. A night where the probabilities are clear-cut is easy to score, a night with many
uncertain epochs is ambiguous (for the machine and often for the scorer too).

**How GSSC chooses the stage (consensus).** GSSC scores the epoch with every combination of the signals
it is given (each chosen EEG alone, the EOG alone, each EEG + the EOG) and, **for each epoch, keeps the
combination that is the most confident** (lowest entropy). It is not a majority vote nor an average.
The hypnodensity is the probabilities of that winning combination, so the stage and its probabilities
follow the same rule.

- **One electrode (fast)**: one EEG + the EOG, 3 combinations.
- **Several electrodes (consensus)**: several EEG + the EOG, more combinations (7 for 3 EEG): slower,
  but a bad electrode on one epoch is outvoted by a better one.

**Inputs**: the EDF files, tool 2's `config_param/remap_reref_persubject.json` (channel names,
reference, and the EOG declared as context channels in tool 2's Section 2bis) and, optionally, the
expert hypnogram remapped by tool 3 (`{id}_Hypnogram_remapped.txt`).

**Outputs**:

```
{data_folder}/
  {id}{output suffix}                     ← automatic hypnogram beside the EDF (tool-3 format: one label per line)
  derivatives/autoscoring_gssc/<subtree>/
    {id}_hypnodensity.tsv                 ← 1 row/epoch: GSSC stage, p_W … p_R, confidence, expert stage
    {id}_scoring_params.json              ← channels, mode, versions (provenance)
  reports_autoscoring_gssc/<subtree>/
    {id}_autoscoring_comparison.tsv       ← agreement metrics with the expert hypnogram (1 row)
    {id}_confusion.tsv                    ← confusion matrix, long format (expert, gssc, n)
    {id}_autoscoring_report.html          ← hypnograms, hypnodensity, confusion matrix
  reports_autoscoring_gssc/
    global_autoscoring_summary.tsv        ← every participant (rebuilt from the per-file tables on disk)
    global_confusion.tsv                  ← pooled confusion matrix
    global_autoscoring_report.html
```

Work through the sections in order: **1** pick the data folder and scan, **2** choose the electrodes,
**3** run.

In [ ]:
try:
    import os
    import json
    import re
    import time
    import datetime
    import warnings
    from pathlib import Path

    import numpy as np
    import pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt

    import mne
    from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, confusion_matrix

    import ipywidgets as widgets
    from IPython.display import display, HTML
    from ipyfilechooser import FileChooser
except ImportError as e:
    print("⚠️ Import error, a package is missing:", e)
else:
    print("✅ Packages imported successfully!")

warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

# GSSC and torch are imported apart: without them the tool cannot score, but the page still renders
# and says how to install them.
# torch >= 2.6 loads model files with weights_only=True by default, which refuses the weights shipped
# inside the gssc package. They come with the installed package (not from an untrusted download), so
# the former behaviour is restored for this process before GSSC loads them.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
GSSC_OK = False
GSSC_MSG = ''
try:
    import torch
    from gssc.infer import EEGInfer
    from gssc.utils import (prepare_inst as gssc_prepare_inst, permute_sigs as gssc_permute_sigs,
                            epo_arr_zscore as gssc_zscore, check_flip_chan as gssc_check_flip_chan,
                            loudest_vote as gssc_loudest_vote)
    from importlib.metadata import version as _pkg_version
    GSSC_VERSION = _pkg_version('gssc')
    TORCH_VERSION = torch.__version__
    GSSC_OK = True
except Exception as e:
    GSSC_VERSION, TORCH_VERSION = None, None
    GSSC_MSG = (
        '<div style="border:2px solid #c0392b;background:#fdecea;padding:8px 12px;border-radius:4px;">'
        f'<b>GSSC is not available</b> ({e}). Install it in the <code>inspect_edf</code> environment, '
        'then restart this page:<br><code>conda install -n inspect_edf -c conda-forge pytorch-cpu</code><br>'
        '<code>python -m pip install gssc==0.0.9</code></div>')
    display(HTML(GSSC_MSG))
if GSSC_OK:
    print(f"✅ GSSC {GSSC_VERSION} (torch {TORCH_VERSION}) ready"
          f"{', GPU available' if torch.cuda.is_available() else ', running on CPU'}.")

# Shared participant handling (tools/participant_selection_lib.py), probed from the repo root (the
# canonical launch folder) or from tools/, so the import works whatever folder Voila is launched from.
import sys
for _p in [os.getcwd(), os.path.join(os.getcwd(), 'tools'),
           os.path.join(os.path.dirname(os.getcwd()), 'tools'), os.path.join(os.getcwd(), '..', 'tools')]:
    if os.path.isfile(os.path.join(_p, 'participant_selection_lib.py')):
        if _p not in sys.path:
            sys.path.insert(0, _p)
        break
import participant_selection_lib as PSL

In [ ]:
EPOCH_SEC = 30
STAGES = ['W', 'N1', 'N2', 'N3', 'R']
GSSC_STAGE = {0: 'W', 1: 'N1', 2: 'N2', 3: 'N3', 4: 'R'}       # GSSC's integer coding
# Clinical hypnogram layout: W on top, then R, then NREM getting deeper downwards.
HYPNO_Y = {'W': 0, 'R': 1, 'N1': 2, 'N2': 3, 'N3': 4}
# Stage colours of the rest of the toolkit (BASE_STAGE_COLORS of tools 0/6/7/9 and qc_rejected_epochs_lib).
STAGE_COLORS = {'W': '#969696', 'N1': '#9e9ac8', 'N2': '#807dba', 'N3': '#6a51a3', 'R': '#c994c7'}
# Hypnogram drawing of tools 6/7: grey step line, REM highlighted in red.
HYPNO_LINE_COLOR = '#555555'
HYPNO_REM_COLOR = '#c0392b'
# Custom (non-AASM) stages declared in config_param/custom_stages.json (written by tool 3): same helper and
# palette as tools 6/7/9 (red is reserved for REM on the hypnogram).
CUSTOM_STAGE_PALETTE = ['#8dd3c7', '#ffffb3', '#bebada', '#80b1d3', '#fdb462', '#b3de69', '#fccde5', '#d9d9d9']


def load_custom_stages(folder):
    """Read config_param/custom_stages.json; return list of kept non-AASM labels ([] if absent/unreadable)."""
    if not folder:
        return []
    path = Path(folder) / 'config_param' / 'custom_stages.json'
    if not path.exists():
        return []
    try:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        stages = data.get('custom_stages', []) if isinstance(data, dict) else []
        return [str(s) for s in stages]
    except Exception:
        return []


def non_aasm_group(label, custom_stages):
    """How an expert label outside W/N1/N2/N3/R is reported: a declared custom stage or MT under its own
    name, anything else (an unexpected label tool 3 did not correct) grouped as 'other'."""
    return label if (label in custom_stages or label == 'MT') else 'other'


def non_aasm_order(groups, custom_stages):
    """Display order of the non-AASM groups: the declared custom stages (in custom_stages.json order),
    then MT, then other."""
    def rank(g):
        if g in custom_stages:
            return (0, custom_stages.index(g))
        return (1, 0) if g == 'MT' else (2, 0)
    return sorted(dict.fromkeys(groups), key=rank)


def non_aasm_color(group, custom_stages):
    """A custom stage keeps the colour tools 6/7 give it (its rank in custom_stages.json); MT and other grey."""
    if group in custom_stages:
        return CUSTOM_STAGE_PALETTE[custom_stages.index(group) % len(CUSTOM_STAGE_PALETTE)]
    return '#bdbdbd'


# Expert hypnogram drawn over a hypnodensity (probability axis 0-1): W on top, N3 at the bottom.
OVERLAY_Y = {'W': 0.9, 'R': 0.7, 'N1': 0.5, 'N2': 0.3, 'N3': 0.1}
CONSENSUS_RULE = 'loudest_vote'   # GSSC's native rule: the most confident combination wins, per epoch
# EOG names that tool 2 may have kept in the remap: never offered as an EEG electrode.
EOG_NAME_RE = re.compile(r'(EOG|^LOC$|^ROC$|^E1$|^E2$)', re.IGNORECASE)


def drop_suffix_duplicates(raw):
    """Keep only the -0 variant when MNE creates -0/-1 duplicates for repeated channel names."""
    groups = {}
    for ch in raw.ch_names:
        if ch.endswith('-0') or ch.endswith('-1'):
            base = ch.rsplit('-', 1)[0]
            groups.setdefault(base, []).append(ch)
    to_drop = []
    for base, ch_list in groups.items():
        if any(c.endswith('-0') for c in ch_list):
            to_drop.extend(c for c in ch_list if not c.endswith('-0'))
    if to_drop:
        raw.drop_channels(to_drop)
    return raw, to_drop


def adapt_remap_dict_to_suffixes(raw, remap_dict):
    """Handle MNE's -0 suffix when the remap config uses the base channel name."""
    ch_set = set(raw.ch_names)
    new_remap = {}
    for base_label, target in remap_dict.items():
        if base_label in ch_set:
            new_remap[base_label] = target
        elif f'{base_label}-0' in ch_set:
            new_remap[f'{base_label}-0'] = target
    return new_remap


def load_json_lenient(path):
    """Strict parse, then repair one trailing comma before a closing } or ] (a common hand edit)."""
    with open(path, encoding='utf-8') as f:
        text = f.read()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return json.loads(re.sub(r',(\s*[}\]])', r'\1', text))


def list_edf(folder):
    return [f for f in sorted(Path(folder).rglob('*'))
            if f.suffix.lower() == '.edf' and not f.name.startswith('._')]


def eeg_options(cfg_entry):
    """Harmonized EEG names usable for scoring: the remap targets minus the reference and the EOG."""
    ref = cfg_entry.get('ref_channels', [])
    ref = ref if isinstance(ref, list) else []
    return [ch for ch in cfg_entry.get('remap', {}).values()
            if ch not in ref and not EOG_NAME_RE.search(ch)]


def load_eeg(edf_path, cfg_entry, wanted):
    """EEG channels `wanted` (harmonized names), re-referenced as declared in tool 2.
    Returns (raw, present) where `present` lists the wanted channels found."""
    raw = mne.io.read_raw_edf(str(edf_path), preload=False, encoding='latin-1',
                              include=list(cfg_entry.get('remap', {}).keys()), verbose=False)
    raw, _ = drop_suffix_duplicates(raw)
    raw.rename_channels(adapt_remap_dict_to_suffixes(raw, cfg_entry.get('remap', {})))
    ref = cfg_entry.get('ref_channels', [])
    present = [ch for ch in wanted if ch in raw.ch_names]
    if not present:
        return None, []
    ref_present = [ch for ch in ref if ch in raw.ch_names] if isinstance(ref, list) else []
    if ref == 'average':
        keep = [ch for ch in raw.ch_names if not EOG_NAME_RE.search(ch)]   # average of every EEG
    else:
        keep = present + [ch for ch in ref_present if ch not in present]
    raw.pick(keep)
    raw.load_data()
    raw.set_channel_types({ch: 'eeg' for ch in raw.ch_names}, verbose=False)
    # GSSC was trained on referenced derivations (e.g. C3-M2): apply tool 2's reference.
    if ref == 'average':
        raw.set_eeg_reference(ref_channels='average', verbose=False)
    elif ref_present:
        raw.set_eeg_reference(ref_channels=ref_present, verbose=False)
    raw.pick(present)
    return raw, present


def load_eog(edf_path, cfg_entry, sfreq, n_times, mode):
    """EOG from tool 2's context channels, at the EEG rate and length.
    Returns (data (n_ch, n_times) in V, names, note)."""
    ctx = cfg_entry.get('context_channels') or {}
    sides = {'EOG-L': ctx.get('eog_left'), 'EOG-R': ctx.get('eog_right')}
    sides = {k: v for k, v in sides.items() if v}
    if mode == 'None':
        return None, [], 'EOG not used (EOG mode = None)'
    if not sides:
        return None, [], 'no EOG declared in tool 2 (context channels): scored on EEG only'
    # Read the EOG apart from the EEG: one read with both would upsample everything to the fastest rate.
    eog = mne.io.read_raw_edf(str(edf_path), preload=False, encoding='latin-1',
                              include=list(sides.values()), verbose=False)
    eog, _ = drop_suffix_duplicates(eog)
    found = {k: v if v in eog.ch_names else f'{v}-0' for k, v in sides.items()}
    found = {k: v for k, v in found.items() if v in eog.ch_names}
    if not found:
        return None, [], f'declared EOG {list(sides.values())} not found in the EDF: scored on EEG only'
    eog.pick(list(found.values()))
    eog.load_data()
    if not np.isclose(eog.info['sfreq'], sfreq):
        eog.resample(sfreq, npad='auto', verbose=False)
    data = eog.get_data()
    # Same length as the EEG (a resample can be off by a sample or two): pad the end with zeros.
    if data.shape[1] >= n_times:
        data = data[:, :n_times]
    else:
        data = np.pad(data, ((0, 0), (0, n_times - data.shape[1])))
    rows = {k: data[i] for i, k in enumerate(found)}
    if mode == 'L − R bipolar' and len(rows) == 2:
        # The L−R derivation doubles the conjugate eye movements and cancels the EEG common to both.
        return (rows['EOG-L'] - rows['EOG-R'])[None, :], ['EOG L-R'], ''
    note = '' if len(rows) == 2 else f'only {list(rows)[0]} found: used alone'
    return np.vstack(list(rows.values())), list(rows), note


def gssc_combination_logits(infer, inst, eeg_names, eog_names):
    """The scoring loop of gssc 0.0.9's EEGInfer.mne_infer, step for step, keeping what mne_infer
    throws away: the network output of EVERY signal combination (mne_infer only returns the final
    stages, not the probabilities). Pinned to gssc 0.0.9: re-check this loop before upgrading gssc.
    Returns (logits (n_combinations, n_epochs, 5), combination names, epoch start times in s)."""
    # GSSC was trained on 0.3-30 Hz data: filter where the data are wider (as mne_infer does)
    band = [0.3 if round(inst.info['highpass'], 2) < 0.3 else None,
            30. if round(inst.info['lowpass'], 2) > 30. else None]
    inst.filter(*band, verbose=False)
    signals = {'eeg': {'chans': eeg_names, 'drop': True, 'flip': False},
               'eog': {'chans': eog_names, 'drop': True, 'flip': False}}
    epo, start_time = gssc_prepare_inst(inst, infer.sig_len, infer.cut)   # 2560-sample (30 s) epochs
    sig_combs, perm_matrix, all_chans, _ = gssc_permute_sigs(epo, signals)
    data = gssc_zscore(epo.get_data(picks=all_chans) * 1e6)
    device = 'cuda' if infer.use_cuda else 'cpu'
    logits, combos = [], []
    for perm in perm_matrix:
        sigs, used = {}, {'eeg': None, 'eog': None}
        for sig_idx, (sig_name, chans) in enumerate(sig_combs.items()):
            if not len(chans[perm[sig_idx]]):
                continue                                  # this signal left out of the combination
            ch, coef = gssc_check_flip_chan(chans[perm[sig_idx]][0])
            sig = coef * data[:, all_chans.index(ch), :infer.sig_len]
            sigs[sig_name] = torch.tensor(sig, dtype=torch.float32).reshape(-1, 1, sig.shape[-1]).to(device)
            used[sig_name] = ch
        with torch.no_grad():
            reps = infer.net(sigs, rep_output='rep_only').swapaxes(-1, 1)
            y, _ = infer.con_net(reps, torch.zeros(10, 1, 256).to(device))   # context network over the night
        logits.append(y[:, 0, ].float().cpu().numpy())
        combos.append(' + '.join(v for v in used.values() if v))
    n_ep = logits[0].shape[0]
    return np.array(logits), combos, np.arange(start_time, n_ep * 30, 30)


def run_gssc(infer, raw_eeg, eog_data, eog_names):
    """One GSSC scoring. Returns (stages (str array), onsets_s, probs (n_epochs, 5),
    winning combination per epoch, number of combinations)."""
    eeg_names = list(raw_eeg.ch_names)
    data = raw_eeg.get_data()
    names, types = list(eeg_names), ['eeg'] * len(eeg_names)
    if eog_data is not None:
        data = np.vstack([data, eog_data])
        names += eog_names
        types += ['eog'] * len(eog_names)
    raw_in = mne.io.RawArray(data, mne.create_info(names, raw_eeg.info['sfreq'], types), verbose=False)
    logits, combos, onsets = gssc_combination_logits(infer, raw_in, eeg_names, eog_names)
    # Consensus = gssc's own loudest_vote (the same call as in mne_infer): per epoch, the combination
    # whose output is the most confident (lowest entropy) wins.
    stage_int = np.asarray(gssc_loudest_vote(logits))
    # Which combination won: the network outputs log-probabilities, so the lowest entropy of
    # loudest_vote is the highest log-probability of the predicted stage.
    win = np.argmax(logits.max(axis=2), axis=0)
    win_logits = logits[win, np.arange(logits.shape[1])]
    e = np.exp(win_logits - win_logits.max(axis=1, keepdims=True))     # softmax -> probabilities
    probs = e / e.sum(axis=1, keepdims=True)
    if not np.array_equal(probs.argmax(axis=1), stage_int):
        raise RuntimeError('hypnodensity inconsistent with the GSSC stages (gssc version changed?)')
    stages = np.array([GSSC_STAGE[int(s)] for s in stage_int])
    return stages, onsets.astype(float), probs, np.array(combos)[win], len(combos)


def load_expert(path):
    """Expert hypnogram, one label per line (tool 3 format)."""
    with open(path, encoding='utf-8', errors='replace') as f:
        return np.array([ln.strip() for ln in f if ln.strip()])


def compare_scoring(expert, gssc):
    """Agreement on the epochs both scored, restricted to expert labels W/N1/N2/N3/R
    (MT, custom stages and '?' are left out and counted)."""
    n = min(len(expert), len(gssc))
    e, g = expert[:n], gssc[:n]
    valid = np.isin(e, STAGES)
    e, g = e[valid], g[valid]
    res = {'n_compared': int(valid.sum()), 'n_excluded_expert': int((~valid).sum())}
    if len(e) == 0:
        res.update({'accuracy': np.nan, 'kappa': np.nan, **{f'f1_{s}': np.nan for s in STAGES}})
        cm = np.zeros((5, 5), dtype=int)
    else:
        res['accuracy'] = accuracy_score(e, g)
        res['kappa'] = cohen_kappa_score(e, g, labels=STAGES)
        f1 = f1_score(e, g, labels=STAGES, average=None, zero_division=np.nan)
        res.update({f'f1_{s}': (f1[i] if s in e or s in g else np.nan) for i, s in enumerate(STAGES)})
        cm = confusion_matrix(e, g, labels=STAGES)
    cm_long = pd.DataFrame([{'expert': STAGES[i], 'gssc': STAGES[j], 'n': int(cm[i, j])}
                            for i in range(5) for j in range(5)])
    return res, cm, cm_long


def _hours(n):
    return np.arange(n + 1) * EPOCH_SEC / 3600


def _hypno_rows(extra):
    """Stage -> row (0 = W at the top ... 4 = N3), then the extra non-AASM labels below N3."""
    rows = dict(HYPNO_Y)
    for i, s in enumerate(extra):
        rows[s] = 5 + i
    return rows


def _step_hypno(ax, labels, title, extra=(), custom_stages=()):
    """Hypnogram as in tools 6/7 (grey line, REM red); `extra` = non-AASM labels drawn as their own rows
    below N3, each highlighted in its custom-stage colour."""
    rows = _hypno_rows(extra)
    y = np.array([rows.get(s, np.nan) for s in labels], dtype=float)
    ax.step(_hours(len(y)), np.append(y, y[-1] if len(y) else np.nan), where='post',
            color=HYPNO_LINE_COLOR, lw=1.2)
    rem = y == HYPNO_Y['R']
    for i in np.where(rem)[0]:
        ax.plot(_hours(len(y))[i:i + 2], [y[i], y[i]], color=HYPNO_REM_COLOR, lw=2.5,
                solid_capstyle='butt')
    for s in extra:
        for i in np.where(y == rows[s])[0]:
            ax.plot(_hours(len(y))[i:i + 2], [y[i], y[i]], lw=2.5, solid_capstyle='butt',
                    color=non_aasm_color(s, list(custom_stages)))
    ax.set_yticks(list(rows.values()))
    ax.set_yticklabels(list(rows.keys()))
    ax.set_ylim(max(rows.values()) + 0.5, -0.5)
    ax.set_title(title, loc='left', fontsize=10)
    ax.grid(axis='y', alpha=0.3)


def plot_hypnograms(gssc, expert, file_id, custom_stages=()):
    """Expert (if any) above GSSC. The expert's non-AASM labels (custom stages, MT, other) get their own rows
    below N3 on BOTH panels, so the two hypnograms keep the same vertical layout."""
    custom_stages = list(custom_stages)
    extra = []
    if expert is not None:
        expert = np.array([s if s in STAGES else non_aasm_group(s, custom_stages) for s in expert])
        extra = non_aasm_order([s for s in expert if s not in STAGES], custom_stages)
    n_rows = 2 if expert is not None else 1
    fig, axes = plt.subplots(n_rows, 1, figsize=(12, (2.0 + 0.3 * len(extra)) * n_rows + 0.4),
                             sharex=True, squeeze=False)
    axes = axes[:, 0]
    if expert is not None:
        _step_hypno(axes[0], expert, 'Expert hypnogram', extra, custom_stages)
    _step_hypno(axes[-1], gssc, 'GSSC hypnogram (automatic)', extra, custom_stages)
    axes[-1].set_xlabel('Time from recording start (h)')
    fig.suptitle(file_id, fontsize=11)
    fig.tight_layout()
    return fig


def _overlay_expert(ax, expert, n_epochs, lw=0.9):
    """Expert hypnogram as a black step line over a hypnodensity (W on top, N3 at the bottom).
    Epochs whose expert label is not W/N1/N2/N3/R (MT, custom, '?') leave a gap. Returns True if drawn."""
    if expert is None:
        return False
    labels = np.asarray(expert)[:n_epochs]
    if not np.isin(labels, STAGES).any():
        return False
    y = np.array([OVERLAY_Y.get(s, np.nan) for s in labels], dtype=float)
    ax.step(_hours(len(y)), np.append(y, y[-1]), where='post', color='black', lw=lw)
    return True


def plot_hypnodensity(probs, file_id, expert=None):
    fig, ax = plt.subplots(figsize=(12, 2.8))
    x = _hours(len(probs))[:-1]
    ax.stackplot(x, probs.T, labels=STAGES, colors=[STAGE_COLORS[s] for s in STAGES],
                 step='post', linewidth=0)
    if _overlay_expert(ax, expert, len(probs)):
        ax_st = ax.twinx()                       # stage names of the expert line, on the right
        ax_st.set_ylim(0, 1)
        ax_st.set_yticks(list(OVERLAY_Y.values()))
        ax_st.set_yticklabels(list(OVERLAY_Y.keys()), fontsize=8)
        ax_st.set_ylabel('Expert (black line)', fontsize=8)
    ax.set_xlim(0, _hours(len(probs))[-1])
    ax.set_ylim(0, 1)
    ax.set_ylabel('Probability')
    ax.set_xlabel('Time from recording start (h)')
    ax.legend(loc='upper left', bbox_to_anchor=(1.06, 1.0), fontsize=9, frameon=False)
    ax.set_title(f'{file_id}: hypnodensity (probability of each stage per epoch)', loc='left', fontsize=10)
    fig.tight_layout()
    return fig


HYPNODENSITY_ROWS_PER_FIG = 20   # participants per stacked figure in the database report


def plot_hypnodensity_stack(nights, x_max_h, title):
    """One hypnodensity per row, all on the same time axis (hours from recording start), for a quick
    visual comparison across participants. nights = list of (row label, probs (n_epochs, 5),
    expert labels or None). The expert hypnogram, when available, is drawn over the row as a black step
    line (W on top, N3 at the bottom)."""
    n = len(nights)
    height = 0.6 * n + 1.0                       # inches: fixed margins below, whatever n
    fig, axes = plt.subplots(n, 1, figsize=(12, height), sharex=True, squeeze=False)
    axes = axes[:, 0]
    has_expert = False
    for ax, (label, probs, expert) in zip(axes, nights):
        x = _hours(len(probs))[:-1]
        ax.stackplot(x, probs.T, colors=[STAGE_COLORS[s] for s in STAGES], step='post', linewidth=0)
        has_expert = _overlay_expert(ax, expert, len(probs), lw=0.7) or has_expert
        ax.set_xlim(0, x_max_h)
        ax.set_ylim(0, 1)
        ax.set_yticks([])
        ax.set_ylabel(label, rotation=0, ha='right', va='center', fontsize=8)
        for side in ('top', 'right', 'left'):
            ax.spines[side].set_visible(False)
    axes[-1].set_xlabel('Time from recording start (h)')
    handles = [plt.Rectangle((0, 0), 1, 1, color=STAGE_COLORS[s]) for s in STAGES]
    fig.legend(handles, STAGES, loc='upper right', bbox_to_anchor=(0.99, 1.0), ncol=5, fontsize=8,
               frameon=False)
    if has_expert:
        title += ' (black line = expert hypnogram, W top to N3 bottom)'
    fig.suptitle(title, x=0.01, ha='left', fontsize=10)
    fig.subplots_adjust(left=0.1, right=0.99, top=1 - 0.45 / height, bottom=0.5 / height, hspace=0.25)
    return fig


def plot_confusion(cm, title):
    """Rows = expert, columns = GSSC. Colour and bold text = % of the expert row, (n) = epoch count."""
    row_sum = cm.sum(axis=1, keepdims=True)
    pct = np.where(row_sum > 0, cm / np.where(row_sum > 0, row_sum, 1) * 100, np.nan)
    fig, ax = plt.subplots(figsize=(5.2, 4.4))
    # Full ColorBrewer Greens (white to #00441b); the text turns white on the dark cells
    greens = plt.cm.Greens
    im = ax.imshow(pct, cmap=greens, vmin=0, vmax=100)
    for i in range(5):
        for j in range(5):
            if row_sum[i, 0] > 0:
                # the % of the expert row is the main reading (comparable across stages and nights);
                # the epoch count below says how many epochs that % rests on.
                # White text where the cell colour is dark (luma of its RGB below 0.5).
                r, g, b, _ = greens(pct[i, j] / 100)
                color = 'white' if 0.2126 * r + 0.7152 * g + 0.0722 * b < 0.5 else '#222222'
                ax.text(j, i - 0.1, f'{pct[i, j]:.0f} %', ha='center', va='center', fontsize=10,
                        fontweight='bold', color=color)
                ax.text(j, i + 0.25, f'({cm[i, j]})', ha='center', va='center', fontsize=7, color=color)
    ax.set_xticks(range(5))
    ax.set_xticklabels(STAGES)
    ax.set_yticks(range(5))
    ax.set_yticklabels(STAGES)
    ax.set_xlabel('GSSC')
    ax.set_ylabel('Expert')
    ax.set_title(title, fontsize=10)
    fig.colorbar(im, ax=ax, fraction=0.046, label='% of the expert stage (row)')
    fig.tight_layout()
    return fig


PROFILE_COLS = [f'p_{s}' for s in STAGES] + [f'pct_gssc_{s}' for s in STAGES]


def non_aasm_profile(expert, gssc, probs, custom_stages):
    """What GSSC makes of the expert epochs outside W/N1/N2/N3/R (custom stages, MT, other): per group, the
    number of epochs, the mean hypnodensity (p_W..p_R) and the % of epochs per GSSC stage. The 5 AASM expert
    stages are added as reference rows (kind 'aasm_reference'), to see which stage a custom one resembles.
    Returns an empty DataFrame when the night has no such epoch."""
    n = min(len(expert), len(gssc))
    e, g, pr = np.asarray(expert[:n]), np.asarray(gssc[:n]), probs[:n]
    if np.isin(e, STAGES).all():
        return pd.DataFrame()
    groups = np.array([s if s in STAGES else non_aasm_group(s, custom_stages) for s in e])
    order = list(STAGES) + non_aasm_order([x for x in groups if x not in STAGES], list(custom_stages))
    rows = []
    for st in order:
        m = groups == st
        if not m.any():
            continue
        kind = ('aasm_reference' if st in STAGES else 'custom' if st in custom_stages
                else 'MT' if st == 'MT' else 'other')
        row = {'stage_expert': st, 'kind': kind, 'n_epochs': int(m.sum())}
        for i, s in enumerate(STAGES):
            row[f'p_{s}'] = float(pr[m, i].mean())
        for s in STAGES:
            row[f'pct_gssc_{s}'] = float((g[m] == s).mean() * 100)
        if kind == 'other':
            row['labels'] = ', '.join(sorted(set(e[m])))
        rows.append(row)
    return pd.DataFrame(rows)


def pool_profiles(df, custom_stages=()):
    """Database profile: per expert stage, the per-night means weighted by the number of epochs."""
    rows = []
    for st, d in df.groupby('stage_expert', sort=False):
        w = d['n_epochs'].astype(float)
        row = {'stage_expert': st, 'kind': d['kind'].iloc[0], 'n_epochs': int(w.sum()),
               'n_nights': int(d['file_id'].nunique())}
        for c in PROFILE_COLS:
            row[c] = float((d[c].astype(float) * w).sum() / w.sum())
        rows.append(row)
    out = pd.DataFrame(rows)
    order = list(STAGES) + non_aasm_order([s for s in out['stage_expert'] if s not in STAGES],
                                          list(custom_stages))
    out['_o'] = [order.index(s) for s in out['stage_expert']]
    return out.sort_values('_o').drop(columns='_o').reset_index(drop=True)


def plot_non_aasm_profile(df, title):
    """Two panels of horizontal stacked bars, one bar per expert stage: the mean hypnodensity (left) and
    the % of epochs per GSSC stage (right). AASM reference bars on top, the non-AASM stages below a line."""
    labels = [f'{r.stage_expert}  (n={r.n_epochs})' for r in df.itertuples()]
    y = np.arange(len(df))
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.45 * len(df) + 1.6), sharey=True)
    for ax, prefix, scale, sub in [(axes[0], 'p_', 1.0, 'Mean hypnodensity (probability)'),
                                   (axes[1], 'pct_gssc_', 100.0, 'Stage given by GSSC (% of epochs)')]:
        left = np.zeros(len(df))
        for s in STAGES:
            v = df[f'{prefix}{s}'].astype(float).values / scale
            ax.barh(y, v, left=left, color=STAGE_COLORS[s], label=s, height=0.7)
            for yi, (l0, vi) in enumerate(zip(left, v)):
                if vi >= 0.12:
                    ax.text(l0 + vi / 2, yi, f'{vi * 100:.0f}%', ha='center', va='center', fontsize=7,
                            color='#222222')
            left += v
        n_ref = int((df['kind'] == 'aasm_reference').sum())
        if 0 < n_ref < len(df):
            ax.axhline(n_ref - 0.5, color='#222222', lw=1)
        ax.set_xlim(0, 1)
        ax.set_title(sub, fontsize=9)
        ax.set_xticks([0, 0.5, 1])
        ax.set_xticklabels(['0', '50 %', '100 %'])
    axes[0].set_yticks(y)
    axes[0].set_yticklabels(labels, fontsize=8)
    axes[0].invert_yaxis()
    axes[0].set_ylabel('Expert stage')
    axes[1].legend(loc='upper left', bbox_to_anchor=(1.0, 1.0), fontsize=8, frameon=False)
    fig.suptitle(title, x=0.01, ha='left', fontsize=10)
    fig.tight_layout()
    return fig


NON_AASM_HELP = ('<p style="font-size:85%;color:#555">GSSC only produces W, N1, N2, N3 and R, so the expert '
                 'epochs scored with another label (custom stages declared in tool 3, MT, other) cannot enter '
                 'the kappa, accuracy, F1 or confusion matrix. This section shows what GSSC makes of them: '
                 'their mean hypnodensity and the stage GSSC gives them, next to the same bars for the five '
                 'AASM expert stages (above the line) as a reference. Example: a custom N4 whose bar looks '
                 'like the N3 bar is read as deep N3 sleep by GSSC.</p>')


def metrics_html(res):
    def f(v, pct=False):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return '–'
        return f'{v * 100:.1f} %' if pct else f'{v:.3f}'
    rows = [('Epochs compared', res['n_compared']),
            ('Epochs left out (expert label not W/N1/N2/N3/R)', res['n_excluded_expert']),
            ('Accuracy', f(res['accuracy'], pct=True)), ("Cohen's kappa", f(res['kappa']))]
    rows += [(f'F1 {s}', f(res[f'f1_{s}'])) for s in STAGES]
    body = ''.join(f'<tr><td style="padding:2px 14px 2px 0">{k}</td><td><b>{v}</b></td></tr>'
                   for k, v in rows)
    return (f'<table style="font-size:90%">{body}</table>'
            '<p style="font-size:85%;color:#555">Kappa: agreement corrected for chance (above 0.6 = '
            'substantial, above 0.8 = almost perfect; two human scorers typically reach 0.7-0.8). '
            'F1 per stage: balance of the epochs of that stage found and correctly labelled.</p>')


def params_html(params):
    rows = ''.join(f'<tr><td style="padding:2px 12px 2px 0"><b>{k}</b></td><td>{v}</td></tr>'
                   for k, v in params.items())
    return f'<table style="font-size:90%">{rows}</table>'


def save_report_html(path, title, items):
    """mne.Report from an ORDERED list of ('fig'|'html', title, payload); None payloads skipped."""
    report = mne.Report(title=title, verbose=False)
    for kind, item_title, payload in items:
        if payload is None:
            continue
        if kind == 'fig':
            report.add_figure(fig=payload, title=item_title, image_format='PNG')
        else:
            report.add_html(html=payload, title=item_title)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    report.save(str(path), overwrite=True, open_browser=False, verbose=False)
    for kind, _, payload in items:
        if kind == 'fig' and payload is not None:
            plt.close(payload)

## 1. Data folder

Select the folder holding the EDF files (searched recursively), then click **Scan**. Tool 2's
`config_param/remap_reref_persubject.json` is read from this folder. The **expert hypnogram suffix** is
detected automatically (leave it empty when there is no expert hypnogram: the nights are then scored
without comparison). The **output suffix** names the automatic hypnogram written beside each EDF.

In [ ]:
fc_data = FileChooser()
fc_data.title = '<b>Select your data folder:</b>'
fc_data.show_only_dirs = True

hypno_suffix = widgets.Text(value='_Hypnogram_remapped.txt', description='Expert hypno suffix:',
                            style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
hypno_suffix_info = widgets.HTML(value='')
out_suffix = widgets.Text(value='_hypnogram_gssc.txt', description='Output hypno suffix:',
                          style={'description_width': '150px'}, layout=widgets.Layout(width='460px'))
btn_scan = widgets.Button(description='Scan', button_style='info', icon='search',
                          layout=widgets.Layout(width='200px', height='34px'))
scan_info = widgets.HTML(value='')
participant_selector = PSL.ParticipantSelector()

STATE = {'cfg': {}, 'edf_by_id': {}, 'not_in_cfg': set(), 'custom': []}


def _detect_hypno_suffix(chooser):
    """Auto-fill the expert hypnogram suffix: among the .txt suffixes found for >= 50 % of the most
    frequent one, prefer the longest (the remapped version). Event exports and the GSSC output are skipped."""
    try:
        if not chooser.selected:
            hypno_suffix_info.value = ''
            return
        folder = Path(chooser.selected)
        edf_files = list_edf(folder)
        if not edf_files:
            hypno_suffix_info.value = '<small style="color:#888;">No EDF file found (recursive scan).</small>'
            return
        all_txt = [f for f in folder.rglob('*') if f.suffix.lower() == '.txt']
        counts = {}
        for edf in edf_files:
            for t in all_txt:
                if os.path.normcase(t.name).startswith(os.path.normcase(edf.stem)):
                    suf = t.name[len(edf.stem):]
                    counts[suf] = counts.get(suf, 0) + 1
        sel = {s: c for s, c in counts.items()
               if 'event' not in s.lower() and 'summary' not in s.lower() and 'studylog' not in s.lower()
               and os.path.normcase(s) != os.path.normcase(out_suffix.value)}
        if not sel:
            hypno_suffix.value = ''
            hypno_suffix_info.value = ('<small style="color:#e67e00;">No expert hypnogram detected: the '
                                       'nights will be scored without comparison.</small>')
            return
        max_count = max(sel.values())
        cands = {s: c for s, c in sel.items() if c >= max_count * 0.5}
        best, best_n = max(cands.items(), key=lambda x: (len(x[0]), x[1]))
        hypno_suffix.value = best
        parts = [f'<b>{s}</b>&nbsp;(×{c}){"&nbsp;← selected" if s == best else ""}'
                 for s, c in sorted(counts.items(), key=lambda x: -x[1])]
        color = '#2e7d32' if best_n == len(edf_files) else '#e67e00'
        hypno_suffix_info.value = (f'<small style="color:{color};">Detected: &nbsp;{"&nbsp;·&nbsp;".join(parts)}'
                                   f'&nbsp;— {best_n}/{len(edf_files)} files matching</small>')
    except Exception as e:
        hypno_suffix_info.value = f'<small style="color:#c0392b;">Error detecting hypnograms: {e}</small>'


fc_data.register_callback(_detect_hypno_suffix)


def output_dirs(data_root, edf_path):
    rel = edf_path.parent.relative_to(data_root)
    return (data_root / 'derivatives' / 'autoscoring_gssc' / rel,
            data_root / 'reports_autoscoring_gssc' / rel)


def output_paths(data_root, edf_path, fid):
    deriv, reports = output_dirs(data_root, edf_path)
    return {'hypno': edf_path.parent / f'{fid}{out_suffix.value}',
            'density': deriv / f'{fid}_hypnodensity.tsv',
            'params': deriv / f'{fid}_scoring_params.json',
            'comparison': reports / f'{fid}_autoscoring_comparison.tsv',
            'confusion': reports / f'{fid}_confusion.tsv',
            'report': reports / f'{fid}_autoscoring_report.html'}


def on_scan(btn):
    btn.disabled = True
    try:
        if not fc_data.selected:
            scan_info.value = '<span style="color:#c0392b;">Select the data folder first.</span>'
            return
        data_root = Path(fc_data.selected)
        cfg_path = data_root / 'config_param' / 'remap_reref_persubject.json'
        if not cfg_path.exists():
            scan_info.value = (f'<span style="color:#c0392b;">{cfg_path} not found: run tool 2 '
                               '(2_select&amp;remap_channels) first.</span>')
            return
        try:
            cfg = load_json_lenient(cfg_path)
        except Exception as e:
            scan_info.value = f'<span style="color:#c0392b;">Cannot read {cfg_path.name}: {e}</span>'
            return
        edf_files = list_edf(data_root)
        cfg_keys = {os.path.normcase(k): k for k in cfg}
        STATE['cfg'] = {}
        STATE['edf_by_id'] = {}
        STATE['not_in_cfg'] = set()
        done, mismatch, n_eog = [], [], 0
        options = []
        for edf in edf_files:
            fid = edf.stem
            STATE['edf_by_id'][fid] = edf
            key = cfg_keys.get(os.path.normcase(fid))
            if key is None:
                STATE['not_in_cfg'].add(fid)
                continue
            STATE['cfg'][fid] = cfg[key]
            ctx = cfg[key].get('context_channels') or {}
            if ctx.get('eog_left') or ctx.get('eog_right'):
                n_eog += 1
            for ch in eeg_options(cfg[key]):
                if ch not in options:
                    options.append(ch)
            # Done = automatic hypnogram + hypnodensity + report all on disk (data written before the report).
            p = output_paths(data_root, edf, fid)
            have = [p['hypno'].exists(), p['density'].exists(), p['report'].exists()]
            if all(have):
                done.append(fid)
            elif any(have):
                mismatch.append(fid)
        participant_selector.set_participants([f.stem for f in edf_files], done=done)
        STATE['custom'] = load_custom_stages(data_root)
        set_eeg_options(options)
        n_cfg = len(STATE['cfg'])
        msg = (f'<span style="color:#2e7d32;">{len(edf_files)} EDF file(s): {n_cfg} configured in tool 2, '
               f'{n_eog} with an EOG declared (context channels), {len(done)} already scored.</span>')
        if STATE['custom']:
            msg += (f'<br><span style="color:#555;">Custom stage(s) from tool 3: {", ".join(STATE["custom"])} '
                    '(shown in a dedicated report section, outside the agreement metrics).</span>')
        if STATE['not_in_cfg']:
            msg += (f'<br><span style="color:#e67e22;">⚠ {len(STATE["not_in_cfg"])} EDF(s) not in the '
                    f'tool-2 JSON, will not be scored: {", ".join(sorted(STATE["not_in_cfg"]))}.</span>')
        if n_cfg and n_eog < n_cfg:
            msg += (f'<br><span style="color:#e67e22;">⚠ {n_cfg - n_eog} configured participant(s) without '
                    'EOG: scored on EEG only (declare the EOG in tool 2, Section 2bis).</span>')
        if mismatch:
            msg += (f'<br><span style="color:#e67e22;">⚠ {len(mismatch)} participant(s) with incomplete '
                    f'outputs, will be rescored: {", ".join(mismatch)}.</span>')
        scan_info.value = msg
    except Exception as e:
        scan_info.value = f'<span style="color:#c0392b;">Scan error: {e}</span>'
    finally:
        btn.disabled = False


btn_scan.on_click(on_scan)

display(fc_data, hypno_suffix, hypno_suffix_info, out_suffix, btn_scan, scan_info,
        widgets.HTML('<b>Participants to score:</b>'), participant_selector.widget)

## 2. Electrodes

**One electrode (fast)**: GSSC scores with that EEG and the EOG. A participant who does not have the
electrode is **not** scored (listed in the log): no other electrode is silently taken instead.
**Several electrodes (consensus)**: GSSC scores with every chosen EEG and the EOG, and keeps the most
confident combination epoch by epoch. A participant missing some of them is scored on the ones present.

The EEG names are the harmonized names of tool 2 (already re-referenced as declared there, e.g. C3-M2).
The EOG comes from the channels declared in tool 2's Section 2bis. GSSC filters the signals itself
(0.3-30 Hz) and resamples them: no preprocessing is needed. On CPU, count about 1-3 min per night.

In [ ]:
tb_mode = widgets.ToggleButtons(options=['One electrode (fast)', 'Several electrodes (consensus)'],
                                value='One electrode (fast)', style={'button_width': '230px'})
dd_eeg = widgets.Dropdown(options=[], description='EEG electrode:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px',
                                                display='' if tb_mode.value.startswith('One') else 'none'))
# Consensus mode: one checkbox per electrode (filled by the scan), plus tick-all / untick-all buttons.
eeg_checks = {}                                   # electrode name -> Checkbox, in the scan's order
box_eeg_checks = widgets.HBox([], layout=widgets.Layout(flex_flow='row wrap', margin='0 0 0 12px'))
btn_eeg_all = widgets.Button(description='Tick all', layout=widgets.Layout(width='110px'))
btn_eeg_none = widgets.Button(description='Untick all', layout=widgets.Layout(width='110px'))
box_consensus = widgets.VBox(
    [widgets.HTML('<b>EEG electrodes:</b>'), box_eeg_checks, widgets.HBox([btn_eeg_all, btn_eeg_none])],
    layout=widgets.Layout(display='none' if tb_mode.value.startswith('One') else ''))


def selected_eeg():
    """Electrodes ticked for the consensus mode, in the scan's order."""
    return [ch for ch, cb in eeg_checks.items() if cb.value]


def _tick_all(value):
    for cb in eeg_checks.values():
        cb.value = value


btn_eeg_all.on_click(lambda _: _tick_all(True))
btn_eeg_none.on_click(lambda _: _tick_all(False))
dd_eog = widgets.Dropdown(options=['L − R bipolar', 'L and R separately', 'None'], value='L − R bipolar',
                          description='EOG:', style={'description_width': '150px'},
                          layout=widgets.Layout(width='360px'))
eeg_hint = widgets.HTML('<small style="color:#888;">Run the scan (Section 1) to list the electrodes.</small>')


def set_eeg_options(options):
    """Fill both EEG pickers after a scan. Default: C3, else C4, else the first electrode, every
    checkbox ticked. A rescan keeps the user's choices for the electrodes still present."""
    previous_one = dd_eeg.value
    previous_ticks = {ch: cb.value for ch, cb in eeg_checks.items()}
    dd_eeg.options = options
    eeg_checks.clear()
    for ch in options:
        eeg_checks[ch] = widgets.Checkbox(value=previous_ticks.get(ch, True), description=ch, indent=False,
                                          layout=widgets.Layout(width='auto', margin='2px 16px 2px 0'))
    box_eeg_checks.children = list(eeg_checks.values())
    if options:
        dd_eeg.value = (previous_one if previous_one in options
                        else next((c for c in ('C3', 'C4') if c in options), options[0]))
        eeg_hint.value = f'<small style="color:#555;">{len(options)} EEG electrode(s) available.</small>'
    else:
        eeg_hint.value = '<small style="color:#c0392b;">No EEG electrode found in the tool-2 JSON.</small>'


def _on_mode(change):
    one = change['new'].startswith('One')
    dd_eeg.layout.display = '' if one else 'none'
    box_consensus.layout.display = 'none' if one else ''


tb_mode.observe(_on_mode, names='value')

display(tb_mode, dd_eeg, box_consensus, eeg_hint, dd_eog)

## 3. Run

Scores the selected participants, writes their outputs, then rebuilds the database summary from every
per-participant table on disk (so a participant scored in an earlier run is still included).

In [ ]:
btn_run = widgets.Button(description='Run automatic scoring', button_style='success', icon='play',
                         layout=widgets.Layout(width='260px', height='38px'))
progress = widgets.IntProgress(value=0, min=0, max=1, description='Progress:',
                               layout=widgets.Layout(width='460px'))
out_run = widgets.Output()


def log(msg):
    with out_run:
        print(msg)


def score_one(infer, data_root, fid, edf_path, cfg_entry, one_mode, wanted, eog_mode):
    """Score one participant and write its outputs. Raises on a fatal error."""
    raw, present = load_eeg(edf_path, cfg_entry, wanted)
    missing = [ch for ch in wanted if ch not in present]
    if one_mode and missing:
        raise RuntimeError(f'electrode {wanted[0]} absent (not substituted)')
    if not present:
        raise RuntimeError(f'none of the chosen electrodes present ({", ".join(wanted)})')
    notes = []
    if missing:
        notes.append(f'electrode(s) absent, scored without them: {", ".join(missing)}')
    eog_data, eog_names, eog_note = load_eog(edf_path, cfg_entry, raw.info['sfreq'], raw.n_times, eog_mode)
    if eog_note:
        notes.append(eog_note)
    for n in notes:
        log(f'[{fid}] ⚠ {n}')

    stages, onsets, probs, winner, n_comb = run_gssc(infer, raw, eog_data, eog_names)

    # Expert hypnogram (optional). GSSC cuts 30-s epochs from the recording start and drops the
    # incomplete tail, like the scorer's epochs: epoch i = line i of the expert file.
    expert = None
    if hypno_suffix.value.strip():
        hp = edf_path.parent / f'{fid}{hypno_suffix.value}'
        if not hp.exists():
            hp = next((p for p in edf_path.parent.iterdir()
                       if os.path.normcase(p.name) == os.path.normcase(hp.name)), hp)
        if hp.exists():
            expert = load_expert(hp)
            if abs(len(expert) - len(stages)) > 1:
                log(f'[{fid}] ⚠ length mismatch: expert {len(expert)} epochs vs GSSC {len(stages)}: '
                    'compared on the common start')
        else:
            notes.append(f'no expert hypnogram ({hp.name}): scored without comparison')
            log(f'[{fid}] no expert hypnogram ({hp.name}): scored without comparison')

    paths = output_paths(data_root, edf_path, fid)
    for d in {paths['density'].parent, paths['report'].parent}:
        d.mkdir(parents=True, exist_ok=True)

    # ---- per-participant data first (the report comes last: report on disk => data on disk) ----
    with open(paths['hypno'], 'w', encoding='utf-8') as f:
        f.write('\n'.join(stages) + '\n')
    dens = pd.DataFrame({'file_id': fid, 'epoch_idx': np.arange(len(stages)), 'onset_s': onsets,
                         'stage_gssc': stages})
    for i, s in enumerate(STAGES):
        dens[f'p_{s}'] = probs[:, i]
    dens['confidence'] = probs.max(axis=1)
    dens['winning_combination'] = winner      # the signals whose scoring was kept for this epoch
    exp_col = np.full(len(stages), '', dtype=object)
    if expert is not None:
        n = min(len(expert), len(stages))
        exp_col[:n] = expert[:n]
    dens['stage_expert'] = exp_col
    # agree is left empty where the expert label is not W/N1/N2/N3/R (MT, custom, '?', not scored)
    dens['agree'] = np.where(~dens['stage_expert'].isin(STAGES), np.nan,
                             (dens['stage_expert'] == dens['stage_gssc']).astype(float))
    dens.to_csv(paths['density'], sep='\t', index=False, float_format='%.4f')

    eog_used = ', '.join(eog_names) if eog_names else 'none'
    params = {'file_id': fid, 'scoring_mode': 'one_electrode' if one_mode else 'consensus',
              'consensus_rule': CONSENSUS_RULE, 'n_combinations': n_comb,
              'hypnodensity': 'softmax of the winning combination (gssc 0.0.9 loop, logits kept)',
              'eeg_used': present, 'eog_used': eog_names, 'eog_mode': eog_mode,
              'reference': cfg_entry.get('ref_channels', []),
              'eog_declared': cfg_entry.get('context_channels') or {},
              'expert_hypnogram_suffix': hypno_suffix.value if expert is not None else None,
              'gssc_version': GSSC_VERSION, 'torch_version': TORCH_VERSION,
              'device': 'cuda' if torch.cuda.is_available() else 'cpu',
              'notes': notes, 'scored_at': datetime.datetime.now().isoformat(timespec='seconds')}
    with open(paths['params'], 'w', encoding='utf-8') as f:
        json.dump(params, f, indent=2, ensure_ascii=False)

    if expert is not None:
        res, cm, cm_long = compare_scoring(expert, stages)
        status = 'compared' if res['n_compared'] else 'no expert W/N1/N2/N3/R epoch'
    else:
        res, cm, cm_long = None, None, None
        status = 'no expert hypnogram'
    row = {'file_id': fid, 'n_epochs_gssc': len(stages),
           'n_epochs_expert': len(expert) if expert is not None else np.nan,
           'n_compared': res['n_compared'] if res else 0,
           'n_excluded_expert': res['n_excluded_expert'] if res else np.nan,
           'accuracy': res['accuracy'] if res else np.nan, 'kappa': res['kappa'] if res else np.nan,
           **{f'f1_{s}': (res[f'f1_{s}'] if res else np.nan) for s in STAGES},
           'mean_confidence': float(probs.max(axis=1).mean()),
           'scoring_mode': params['scoring_mode'], 'eeg_used': ', '.join(present), 'eog_used': eog_used,
           'status': status}
    pd.DataFrame([row]).to_csv(paths['comparison'], sep='\t', index=False, float_format='%.4f')
    # Expert epochs outside W/N1/N2/N3/R: what GSSC makes of them (only when the night has some).
    custom_path = paths['comparison'].parent / f'{fid}_custom_stages.tsv'
    profile = (non_aasm_profile(expert, stages, probs, STATE['custom']) if expert is not None
               else pd.DataFrame())
    if len(profile):
        profile.insert(0, 'file_id', fid)
        profile.to_csv(custom_path, sep='\t', index=False, float_format='%.4f')
    elif custom_path.exists():
        custom_path.unlink()                 # a former profile no longer valid
    if cm_long is not None:
        cm_long.insert(0, 'file_id', fid)
        cm_long.to_csv(paths['confusion'], sep='\t', index=False)
    elif paths['confusion'].exists():
        paths['confusion'].unlink()          # a former comparison no longer valid

    # ---- report ----
    shown = {'EEG used': ', '.join(present), 'EOG used': eog_used,
             'Mode': f'{params["scoring_mode"]} ({n_comb} signal combinations, rule: most confident per epoch)',
             'Reference (tool 2)': params['reference'], 'GSSC / torch': f'{GSSC_VERSION} / {TORCH_VERSION}',
             'Epochs scored': len(stages), 'Mean confidence': f'{row["mean_confidence"]:.2f}'}
    if notes:
        shown['Notes'] = '<br>'.join(f'⚠ {n}' for n in notes)
    items = [('html', 'Scoring parameters', params_html(shown)),
             ('fig', 'Hypnograms', plot_hypnograms(stages, expert, fid, STATE['custom'])),
             ('fig', 'Hypnodensity', plot_hypnodensity(probs, fid, expert))]
    if res is not None and res['n_compared']:
        items += [('html', 'Agreement with the expert', metrics_html(res)),
                  ('fig', 'Confusion matrix', plot_confusion(
                      cm, f'{fid}: kappa {res["kappa"]:.2f}, accuracy {res["accuracy"] * 100:.0f} %'))]
    else:
        items += [('html', 'Agreement with the expert',
                   f'<p>No comparison: {status}.</p>')]
    if len(profile):
        items += [('html', 'Expert stages outside W/N1/N2/N3/R', NON_AASM_HELP),
                  ('fig', 'Expert stages outside W/N1/N2/N3/R: what GSSC makes of them',
                   plot_non_aasm_profile(profile, f'{fid}: expert stages outside W/N1/N2/N3/R, '
                                                  'with the AASM stages as reference'))]
    save_report_html(paths['report'], f'Automatic sleep scoring (GSSC): {fid}', items)
    return row


def rebuild_database_summary(data_root):
    """Database tables + report, rebuilt from every per-participant table on disk."""
    rep_root = data_root / 'reports_autoscoring_gssc'
    if not rep_root.exists():
        return
    # Per-participant tables only: the database tables written here (global_*) share their suffixes
    # (global_confusion.tsv, global_custom_stages.tsv) and would otherwise be counted again.
    def per_file(pattern):
        return sorted(p for p in rep_root.rglob(pattern) if not p.name.startswith('global_'))

    comps = per_file('*_autoscoring_comparison.tsv')
    if not comps:
        return
    summ = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in comps], ignore_index=True)
    summ = summ.sort_values('file_id')
    summ.to_csv(rep_root / 'global_autoscoring_summary.tsv', sep='\t', index=False, float_format='%.4f')
    confs = per_file('*_confusion.tsv')
    items = []
    if confs:
        allc = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in confs], ignore_index=True)
        pooled = allc.groupby(['expert', 'gssc'], as_index=False)['n'].sum()
        pooled.to_csv(rep_root / 'global_confusion.tsv', sep='\t', index=False)
        cm = (pooled.pivot(index='expert', columns='gssc', values='n')
              .reindex(index=STAGES, columns=STAGES).fillna(0).astype(int).values)
        n_tot = cm.sum()
        acc = np.trace(cm) / n_tot if n_tot else np.nan
        items.append(('fig', 'Pooled confusion matrix', plot_confusion(
            cm, f'All participants pooled ({allc["file_id"].nunique()} nights, {n_tot} epochs), '
                f'accuracy {acc * 100:.0f} %')))
    customs = per_file('*_custom_stages.tsv')
    if customs:
        allp = pd.concat([pd.read_csv(p, sep='\t', dtype={'file_id': str}) for p in customs], ignore_index=True)
        pooled_p = pool_profiles(allp, STATE['custom'])
        pooled_p.to_csv(rep_root / 'global_custom_stages.tsv', sep='\t', index=False, float_format='%.4f')
        items.append(('html', 'Expert stages outside W/N1/N2/N3/R',
                      NON_AASM_HELP + f'<p style="font-size:85%;color:#555">Pooled over the '
                      f'{allp["file_id"].nunique()} night(s) that have such epochs (reference bars from the '
                      'same nights).</p>'))
        items.append(('fig', 'Expert stages outside W/N1/N2/N3/R: what GSSC makes of them',
                      plot_non_aasm_profile(pooled_p, 'All nights pooled: expert stages outside W/N1/N2/N3/R')))
    elif (rep_root / 'global_custom_stages.tsv').exists():
        (rep_root / 'global_custom_stages.tsv').unlink()
    comp = summ[summ['n_compared'] > 0]
    if len(comp):
        fig, ax = plt.subplots(figsize=(7, 3.2))
        for i, col in enumerate(['kappa', 'accuracy']):
            vals = comp[col].astype(float).values
            ax.scatter(np.full(len(vals), i) + np.random.default_rng(0).uniform(-0.12, 0.12, len(vals)),
                       vals, s=18, color='#3a87c8', alpha=0.8)
            ax.plot([i - 0.25, i + 0.25], [np.nanmedian(vals)] * 2, color='#222222', lw=2)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Cohen's kappa", 'Accuracy'])
        ax.set_ylim(0, 1)
        ax.set_title('Agreement with the expert per participant (line = median)', fontsize=10)
        ax.grid(axis='y', alpha=0.3)
        fig.tight_layout()
        items.append(('fig', 'Agreement per participant', fig))
    # Every participant's hypnodensity stacked on one time axis, for a quick visual comparison.
    dens_files = sorted((data_root / 'derivatives' / 'autoscoring_gssc').rglob('*_hypnodensity.tsv'),
                        key=lambda p: p.name)
    kappa_by_id = dict(zip(summ['file_id'].astype(str), pd.to_numeric(summ['kappa'], errors='coerce')))
    nights = []
    for p in dens_files:
        try:
            d = pd.read_csv(p, sep='\t', dtype={'file_id': str}, keep_default_na=False)
            fid = str(d['file_id'].iloc[0])
            k = kappa_by_id.get(fid, np.nan)
            label = fid if pd.isna(k) else f'{fid}\nκ {k:.2f}'
            expert = d['stage_expert'].astype(str).values if 'stage_expert' in d else None
            if expert is not None and not (expert != '').any():
                expert = None                                # scored without an expert hypnogram
            nights.append((label, d[[f'p_{s}' for s in STAGES]].astype(float).values, expert))
        except Exception as e:
            log(f'⚠ Hypnodensity not shown in the database report ({p.name}): {e}')
    if nights:
        x_max = max(len(pr) for _, pr, _ in nights) * EPOCH_SEC / 3600
        n_per = HYPNODENSITY_ROWS_PER_FIG
        for start in range(0, len(nights), n_per):
            part = nights[start:start + n_per]
            title = (f'Hypnodensities, participants {start + 1}-{start + len(part)} of {len(nights)}'
                     if len(nights) > n_per else f'Hypnodensities ({len(nights)} participants)')
            items.append(('fig', title, plot_hypnodensity_stack(part, x_max, title)))
    cols = ['file_id', 'n_epochs_gssc', 'n_compared', 'accuracy', 'kappa', 'mean_confidence',
            'scoring_mode', 'eeg_used', 'eog_used', 'status']
    items.append(('html', 'Participants', summ[[c for c in cols if c in summ]].to_html(
        index=False, float_format=lambda v: f'{v:.3f}', na_rep='–')))
    save_report_html(rep_root / 'global_autoscoring_report.html', 'Automatic sleep scoring (GSSC): database',
                     items)


def run_scoring(btn):
    btn.disabled = True
    out_run.clear_output()
    try:
        if not GSSC_OK:
            with out_run:
                display(HTML(GSSC_MSG))
            return
        if not fc_data.selected or not STATE['cfg']:
            log('Select the data folder and click Scan (Section 1) first.')
            return
        if hypno_suffix.value.strip() and \
                os.path.normcase(hypno_suffix.value.strip()) == os.path.normcase(out_suffix.value.strip()):
            log('⛔ The output suffix equals the expert hypnogram suffix: the expert hypnograms would be '
                'overwritten. Change the output suffix.')
            return
        if not out_suffix.value.strip().lower().endswith('.txt'):
            log('⛔ The output suffix must end with .txt.')
            return
        one_mode = tb_mode.value.startswith('One')
        wanted = [dd_eeg.value] if one_mode else selected_eeg()
        if not wanted or wanted == [None]:
            log('⛔ Choose at least one EEG electrode (Section 2).')
            return
        data_root = Path(fc_data.selected)
        selected = participant_selector.selected_ids()
        if participant_selector.warning_html():
            with out_run:
                display(HTML(participant_selector.warning_html()))
        not_cfg = [f for f in selected if f in STATE['not_in_cfg']]
        if not_cfg:
            log(f'⚠ Not in the tool-2 JSON, not scored: {", ".join(not_cfg)}')
        selected = [f for f in selected if f in STATE['cfg']]
        log(f'{len(selected)} participant(s) to score, EEG: {", ".join(wanted)}, EOG: {dd_eog.value}.')

        failed, attempted = [], set()
        progress.max = max(len(selected), 1)
        progress.value = 0
        infer = None
        if selected:
            try:
                infer = EEGInfer(use_cuda=torch.cuda.is_available())
            except Exception as e:
                log(f'⛔ Cannot load the GSSC model: {e}')
                return
        for i, fid in enumerate(selected):
            attempted.add(fid)
            t0 = time.time()
            try:
                row = score_one(infer, data_root, fid, STATE['edf_by_id'][fid], STATE['cfg'][fid],
                                one_mode, wanted, dd_eog.value)
                kappa = '' if pd.isna(row['kappa']) else f', kappa {row["kappa"]:.2f}'
                log(f'[{fid}] ✅ {row["n_epochs_gssc"]} epochs scored{kappa} ({time.time() - t0:.0f} s)')
            except Exception as e:
                log(f'[{fid}] ❌ {e}')
                failed.append({'file_id': fid, 'reason': str(e)})
            plt.close('all')
            progress.value = i + 1

        # failed_autoscoring.tsv: this run's failures + the earlier ones of participants not retried.
        try:
            rep_root = data_root / 'reports_autoscoring_gssc'
            fpath = rep_root / 'failed_autoscoring.tsv'
            old = pd.read_csv(fpath, sep='\t', dtype={'file_id': str}) if fpath.exists() else pd.DataFrame()
            if len(old):
                tried = {os.path.normcase(f) for f in attempted}
                old = old[~old['file_id'].map(os.path.normcase).isin(tried)]
            allf = pd.concat([old, pd.DataFrame(failed)], ignore_index=True)
            if len(allf):
                rep_root.mkdir(parents=True, exist_ok=True)
                allf.to_csv(fpath, sep='\t', index=False)
            elif fpath.exists():
                fpath.unlink()
        except Exception as e:
            log(f'⚠ Could not update failed_autoscoring.tsv: {e}')

        try:
            rebuild_database_summary(data_root)
            log(f'Database summary rebuilt: {data_root / "reports_autoscoring_gssc"}')
        except Exception as e:
            log(f'⚠ Could not rebuild the database summary: {e}')
        log(f'Done: {len(selected) - len(failed)} scored, {len(failed)} failed.')
    except Exception as e:
        log(f'❌ Unexpected error: {e}')
    finally:
        btn.disabled = False


btn_run.on_click(run_scoring)
display(btn_run, progress, out_run)